# Exercise 3 · Commit and push a change by hand

You will change your copy of the starter on your computer, commit the change, and push it to GitHub.

**Goal:** Clone your copy, change `README.md`, commit it, push it, and find the commit on github.com.

**Why it matters:** You hand in challenge 1 by pushing commits. The agent will do this for you later, so you need to know each move.

**Before you start:**
- You did exercise 1 today: you have your own copy of the starter on GitHub, and its link.
- Step 2 clones that copy into `practice/starter` with one command: `git clone <your link> starter`.
- `git push` needs your GitHub sign-in in this terminal. One documented way is `gh auth login`: choose `HTTPS`, then enter `Y` to authenticate Git ([Caching your GitHub credentials](https://docs.github.com/en/get-started/git-basics/caching-your-github-credentials-in-git)). If your trainer gives another way, use that. Your GitHub password does not work there.

**What the check measures:** `practice/starter` is a clone of your copy, your newest commit changes `README.md`, and GitHub has that commit.

**Done when:** the last cell prints `Done:`.

## The situation

Your copy of the starter is on GitHub. You add one line to its `README.md` on your computer. Then you send it to GitHub, as you will with your challenge work. The line is made up.

## Step 1 · Write your link

Run the cell with your link from exercise 1. It makes `practice/` and prints the clone command for Step 2. It does not go online.

Expect a line that starts with `git clone https://github.com/`.

In [ ]:
# Step 1: paste the link you handed in for exercise 1 between the quotes, then run this cell
MY_REPO = ""

import re
from pathlib import Path

PRACTICE = Path("practice")
PRACTICE.mkdir(exist_ok=True)
found = re.search(r"github\.com/([\w.-]+)/([\w.-]+)", MY_REPO)
if (PRACTICE / "starter").exists():
    print("Kept practice/starter: it is already there. Go to Step 3.")
elif not found:
    print("MY_REPO is empty or not a GitHub link. Paste your link from exercise 1, then run this cell again.")
else:
    print("In Step 2, clone your copy with this line:")
    print(f"git clone https://github.com/{found[1]}/{found[2].removesuffix('.git')}.git starter")

## Step 2 · Clone your copy

Choose **Terminal > New Terminal**. Type each line, then press Enter. For the third line, paste the line that Step 1 printed:

```sh
source .venv/bin/activate
cd week-02/day-02/exercise-03-commit-and-push/practice
git clone https://github.com/<your-account>/<name>.git starter
cd starter
git log --oneline
```

Expect `Cloning into 'starter'...`, then one commit: your copy started with a single commit. Git names the repository you cloned from `origin`: here, your copy on GitHub.

## Step 3 · Change README.md

```sh
echo "My copy of the challenge 1 starter." >> README.md
git status
```

Expect `modified:   README.md`.

### Learn: commit, then push

`git add` picks the change for the next commit. `git commit` saves it in your clone's history ([git-commit](https://git-scm.com/docs/git-commit)). A commit stays on your computer. `git push` sends your new commits to GitHub ([git-push](https://git-scm.com/docs/git-push)).

## Step 4 · Commit the change

```sh
git add README.md
git commit -m "Add a line to README"
git log --oneline
```

Expect two lines, newest first: `Add a line to README`, then your copy's first commit.

## Step 5 · Push it to GitHub

```sh
git push
```

Expect a line that ends with `main -> main`. If Git asks for a password, stop and ask your trainer: you need the GitHub sign-in first.

## Step 6 · Find it on github.com

Open your copy on github.com. Reload the page.

Expect your line at the end of the README shown below the file list. Next to `README.md`, expect `Add a line to README`.

<details><summary>Show the answer: you see no change. What went wrong?</summary>

The commit is not pushed yet, or you opened the course's starter. Why: GitHub shows only pushed commits. Run `git status`: `Your branch is ahead of 'origin/main'` means push again (Step 5). Check that the address bar shows your account.

</details>

## Start over (only if you need it)

In the terminal, run `cd ..` to go to `practice/`. Then run `mv starter starter-old` to rename the clone. Then do Step 2 again. Your pushed commits stay on GitHub.

## Check your work

Run the cell. Expect three `ok:` lines, then `Done:`. A `Not yet:` line names the first problem and what to do. The last check asks GitHub, so it needs the internet.

In [ ]:
# Check your work: read practice/starter, then ask GitHub for its newest commit (a public read)
import os
import re
import subprocess
from pathlib import Path

CLONE = Path("practice") / "starter"
NO_PROMPT = dict(os.environ, GIT_TERMINAL_PROMPT="0")


def git(*args) -> str:
    """Return git's output in practice/starter, or "" if the command fails."""
    try:
        done = subprocess.run(["git", "-C", str(CLONE), *args], capture_output=True, text=True,
                              timeout=60, env=NO_PROMPT)
    except (OSError, subprocess.TimeoutExpired):
        return ""
    return done.stdout.strip() if done.returncode == 0 else ""


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    if not (CLONE / ".git").exists():
        return "practice/starter is missing. Clone your copy into it (Step 2)."
    url = git("config", "--get", "remote.origin.url")
    owner = re.search(r"github\.com[/:]([\w.-]+)/", url)
    if not owner:
        return f"practice/starter came from {url!r}, not from github.com. Clone your copy again (Start over, Step 2)."
    if owner[1].lower() == "gitrolltraining":
        return "practice/starter is the course's starter, not your copy. Clone your own link (Start over, Step 2)."
    print(f"ok: practice/starter is a clone of your copy ({owner[1]})")
    changed = git("diff-tree", "--no-commit-id", "--name-only", "-r", "HEAD").splitlines()
    if git("rev-list", "--count", "HEAD") in ("", "0", "1") or "README.md" not in changed:
        return "your newest commit does not change README.md. Change it and commit (Step 3 and Step 4)."
    print("ok: your newest commit changes README.md")
    remote = git("ls-remote", "origin", "HEAD").split()
    if not remote:
        return "Git could not read your copy on GitHub. Check your internet connection, then run this cell again."
    if remote[0] != git("rev-parse", "HEAD"):
        return "GitHub does not have your newest commit. Push it (Step 5)."
    print("ok: GitHub has your newest commit")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = f"the check stopped on {type(error).__name__}: {error}. Do Step 2, then run this cell."
print(f"Not yet: {problem}" if problem else
      "Done: you committed a change by hand and pushed it to your copy on GitHub.")

## What you learned

- `git add` and `git commit` save a change on your computer. `git push` sends it to GitHub.
- GitHub shows only what you pushed. `git status` tells you when a commit is not pushed yet.

**Reflect:** Challenge 1 is graded from your copy on GitHub. What happens to work you commit but never push?